# 01 — Data Cleaning (Mental Health)

Ce notebook reprend **étape par étape** les actions du script `src/preprocessing/cleaning.py` (classe `DataCleaner`), avec pour chaque étape :

- **Action** : ce que fait le code
- **Justification** : pourquoi on le fait, et ce que montrent les résultats

**Ordre du pipeline :** chargement → inspection → nettoyage du texte → conversion des types → valeurs manquantes → outliers → doublons → validation → sauvegarde.

## 0. Imports et chemins

In [4]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

RAW_PATH = "../data/raw/student_mental_health.csv"
CLEAN_PATH = "../data/processed/dataset_clean.csv"

## 1. Chargement du dataset

**Action :** lecture du CSV brut avec `pd.read_csv`.


In [24]:
df = pd.read_csv(RAW_PATH)
df.head()

,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium
3,STU0004,24,Male,1,4.5,14.8,2.23,5.5,7.6,High,3.7,8.0,73.9,No,Average,Yes,High
4,STU0005,20,Female,1,6.7,17.2,2.69,4.3,7.0,Moderate,8.0,7.4,99.8,Yes,NaN,No,Medium


## 2. STRUCTURE DU DATASET

**Action :** Aficher le structure du dataset


In [6]:
print("Dimensions :", df.shape)

print("\nColonnes :")
print(df.columns.tolist())

print("\nTypes et valeurs non nulles :")
df.info()

print("\nStatistiques numériques :")
print(df.describe())

Dimensions : (2000, 17)

Colonnes :
['Student_ID', 'Age', 'Gender', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Physical_Activity', 'Financial_Stress', 'Family_Support', 'Attendance_Rate', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History', 'Mental_Health_Risk']

Types et valeurs non nulles :
<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Student_ID                      2000 non-null   str    
 1   Age                             2000 non-null   int64  
 2   Gender                          2000 non-null   str    
 3   Year_of_Study                   2000 non-null   int64  
 4   Sleep_Hours                     1938 non-null   float64
 5   Study_Hours_Per_Week            2000 non-null   float64
 6   GPA                             1962 no

## 3. VALEURS MANQUANTES 
**Action :** afficher les valeurs manquantes


In [26]:
missing = df.isnull().sum()

missing
print(missing[missing > 0].sort_values(ascending=False))

Financial_Stress      91
Diet_Quality          69
Sleep_Hours           62
Social_Media_Hours    58
Family_Support        54
Attendance_Rate       47
GPA                   38
dtype: int64


## 4. DOUBLONS

**Action :** aficher le number DOUBLONS

In [12]:
duplicates = df.duplicated().sum()

print("\n DOUBLONS ")
print("Nombre de doublons :", duplicates)


 DOUBLONS 
Nombre de doublons : 0


## 5. Categorical values

In [13]:
categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    print(f"\n===== {col} =====")
    print(df[col].value_counts(dropna=False).head(20))


===== Student_ID =====
Student_ID
STU0001    1
STU0002    1
STU0003    1
STU0004    1
STU0005    1
STU0006    1
STU0007    1
STU0008    1
STU0009    1
STU0010    1
STU0011    1
STU0012    1
STU0013    1
STU0014    1
STU0015    1
STU0016    1
STU0017    1
STU0018    1
STU0019    1
STU0020    1
Name: count, dtype: int64

===== Gender =====
Gender
Female        971
Male          932
Non-binary     97
Name: count, dtype: int64

===== Physical_Activity =====
Physical_Activity
Moderate    906
Low         679
High        415
Name: count, dtype: int64

===== Part_Time_Job =====
Part_Time_Job
No     1289
Yes     711
Name: count, dtype: int64

===== Diet_Quality =====
Diet_Quality
Average    860
Poor       587
Good       484
NaN         69
Name: count, dtype: int64

===== Previous_Mental_Health_History =====
Previous_Mental_Health_History
No     1408
Yes     592
Name: count, dtype: int64

===== Mental_Health_Risk =====
Mental_Health_Risk
High      680
Medium    660
Low       660
Name: count, dt

/tmp/ipykernel_41478/731942264.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include="object").columns


## Replace with a NAN value

In [30]:
df.columns = df.columns.str.strip()

for col in df.select_dtypes("object"):
    df[col] = df[col].str.strip()

df = df.replace(["NaN", "nan"], np.nan)
df = df.replace(r"^\s*$", np.nan, regex=True)

/tmp/ipykernel_41478/1953194414.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes("object"):


In [35]:
missing = df.isnull().sum()


df.info()
missing

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Student_ID                      2000 non-null   str    
 1   Age                             2000 non-null   int64  
 2   Gender                          2000 non-null   str    
 3   Year_of_Study                   2000 non-null   int64  
 4   Sleep_Hours                     1938 non-null   float64
 5   Study_Hours_Per_Week            2000 non-null   float64
 6   GPA                             1962 non-null   float64
 7   Social_Media_Hours              1942 non-null   float64
 8   Screen_Time_Hours               2000 non-null   float64
 9   Physical_Activity               2000 non-null   str    
 10  Financial_Stress                1909 non-null   float64
 11  Family_Support                  1946 non-null   float64
 12  Attendance_Rate                 1953 non-null

Student_ID                         0
Age                                0
Gender                             0
Year_of_Study                      0
Sleep_Hours                       62
Study_Hours_Per_Week               0
GPA                               38
Social_Media_Hours                58
Screen_Time_Hours                  0
Physical_Activity                  0
Financial_Stress                  91
Family_Support                    54
Attendance_Rate                   47
Part_Time_Job                      0
Diet_Quality                      69
Previous_Mental_Health_History     0
Mental_Health_Risk                 0
dtype: int64

## Determine outliers

In [14]:
numeric_cols = df.select_dtypes(include="number").columns

print(numeric_cols)

Index(['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA',
       'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress',
       'Family_Support', 'Attendance_Rate'],
      dtype='str')


In [29]:
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    
    print(f"\n===== {col} =====")
    print(f"Q1       : {Q1:.2f}")
    print(f"Q3       : {Q3:.2f}")
    print(f"IQR      : {IQR:.2f}")
    print(f"Lower    : {lower:.2f}")
    print(f"Upper    : {upper:.2f}")
    print(f"Outliers : {outliers}")


===== Unnamed: 0 =====
Q1       : 191.75
Q3       : 575.25
IQR      : 383.50
Lower    : -383.50
Upper    : 1150.50
Outliers : 0

===== Pregnancies =====
Q1       : 1.00
Q3       : 6.00
IQR      : 5.00
Lower    : -6.50
Upper    : 13.50
Outliers : 4

===== Glucose =====
Q1       : 99.00
Q3       : 141.00
IQR      : 42.00
Lower    : 36.00
Upper    : 204.00
Outliers : 0

===== BloodPressure =====
Q1       : 64.00
Q3       : 80.00
IQR      : 16.00
Lower    : 40.00
Upper    : 104.00
Outliers : 14

===== SkinThickness =====
Q1       : 22.00
Q3       : 36.00
IQR      : 14.00
Lower    : 1.00
Upper    : 57.00
Outliers : 3

===== Insulin =====
Q1       : 76.25
Q3       : 190.00
IQR      : 113.75
Lower    : -94.38
Upper    : 360.62
Outliers : 24

===== BMI =====
Q1       : 27.50
Q3       : 36.60
IQR      : 9.10
Lower    : 13.85
Upper    : 50.25
Outliers : 8

===== DiabetesPedigreeFunction =====
Q1       : 0.24
Q3       : 0.63
IQR      : 0.38
Lower    : -0.33
Upper    : 1.20
Outliers : 29

===== A

In [30]:
df.isnull().sum()

Unnamed: 0                    0
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
dtype: int64

## export result

In [31]:
result = df.to_csv(CLEAN_PATH, index=False)